# Exercise Session: Where Should a Global Health Foundation Invest First?

> ## PART 1 · TASK DESCRIPTION (for students)
> You work on this part. Part 2 of this notebook is the reference solution and is **not** distributed.

**Format (90 min):** 10 min briefing · **30 min group work** · 5 × 8 min presentations incl. questions · 10 min debrief.
**Tools:** Python, scikit-learn, and **an AI coding assistant of your choice**. You are expected to use it.

## 1. The case

**Horizon Health & Development Foundation** is a mid-sized private foundation. Its board has just approved a new
programme: **€60 million over three years** for improving living conditions in low- and middle-income countries.

The foundation cannot support every country. Experience from earlier programmes shows that a grant below roughly
**€4 million** has no measurable effect, so the **first funding wave can cover at most ~15 countries**.
Every decision will be scrutinised by donors and the press, so the board must be able to *explain* why a country was
chosen, and why another one was not.

You have been hired as data scientist by the programme director, Dr. Amara Okafor. In a short e-mail she wrote:

> *"The board meets next week. I don't need another ranking of countries by GDP. Three things worry me:*
> *(1) we all 'know' which countries are in trouble, but I can't back that up with evidence;*
> *(2) our past programmes treated all poor countries alike. A child-health campaign that works in one place was*
> *irrelevant in another. I suspect there are different types of countries needing different types of help;*
> *(3) I have to defend the shortlist against critics. Tell me how robust it is, and where you are unsure."*

The programme team can currently offer three types of programmes, and each country receives **one** of them:

| Programme | Focus |
|---|---|
| **A · Child survival & primary care** | Vaccination, maternal and child health services, nutrition |
| **B · Health system strengthening** | Health financing, supply chains, workforce; for systems that depend on imports and external funding |
| **C · Economic resilience** | Livelihoods, trade, price stability, public finance |

## 2. Your mandate

Prepare a recommendation for the board that answers:

1. **Need.** Which countries have the greatest need for support, and what does "need" mean in the data?
2. **Types.** Are the world's countries a homogeneous mass, or do distinct *types* of countries exist? What distinguishes them?
3. **Allocation.** Which **≤15 countries** should receive funding in the first wave, and which programme (A, B or C) fits each?
4. **Confidence.** How robust is this recommendation? Which countries are borderline, and what are the limits of your data?

There are no labels in this problem. Nobody has told us which countries "are" in need or which "type" they belong to.
Deciding how to get from raw indicators to defensible answers is **part of the task**.

## 3. Data

Socio-economic and health indicators for 167 countries (one row per country, public data, ca. 2010 to 2015, no missing values).
File: `country_data.csv` (data dictionary: `country_data_dictionary.csv`).

| Variable | Description |
|---|---|
| `country` | Country name |
| `child_mort` | Deaths of children under 5 per 1,000 live births |
| `exports` | Exports of goods and services, % of GDP |
| `imports` | Imports of goods and services, % of GDP |
| `health` | Total health spending, % of GDP |
| `income` | Net income per person (USD) |
| `inflation` | Annual growth rate of total GDP (%) as a measure of inflation |
| `life_expec` | Life expectancy at birth (years) if current mortality patterns persist |
| `total_fer` | Children per woman if current fertility rates persist |
| `gdpp` | GDP per capita (USD) |

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/Humboldt-WI/bads/master/excercise-cases/country_data.csv"
#df = pd.read_csv(url, index_col="country")
df = pd.read_csv("country_data.csv", index_col="country")

df.head()

## 4. Ground rules for working with AI

* AI assistants will write most of your code. **Your job is to decide *what* should be computed and *why*, and to judge whether the result makes sense.**
* A plot or a number that an AI produced is not a finding. Every claim in your presentation needs a sentence on *what it means for the decision*.
* Keep a short **AI log** (3 to 4 bullets): your most useful prompt, one thing the AI got wrong or did not question, and one thing you changed after checking.

## 5. Before you code: think (5 min)

Close your laptop, or at least your AI tool, and write down on paper:

* The board's questions are not data science questions yet. **What would you have to compute or find out to answer each one?**
* What could go wrong if you feed this data to an algorithm untouched?
* Which of your answers can the data give, and which require judgement that you must explicitly state?

Only then start working.

## 6. Deliverable: a 5-minute talk (max. 3 slides or one annotated notebook section)

1. **Recommendation:** your shortlist with programme type, in a single table or figure.
2. **Evidence:** the 1 or 2 analyses that convince a non-technical board member, with plain-language interpretation.
3. **Confidence and caveats:** what could make the board wrong to trust you.

Be prepared to answer: *"Why these countries and not others?"* and *"Why did you choose that number of groups?"*

Use the cells below for your work.

In [ ]:
# Your work starts here

<br><br>

---
---

# PART 2 · REFERENCE SOLUTION AND INSTRUCTOR NOTES


---
---

## 7. From business problem to data science tasks

The students should discover this decomposition themselves. The table is the target against which to compare their approaches.

| Business question | Data science sub-task | Unsupervised learning technique | Why it matters |
|---|---|---|---|
| (0) Can we trust the data? | Audit distributions, scales, outliers, special cases | Histograms, summary stats | Skew, extreme values (e.g. exports > 100% of GDP) and different units would dominate any distance-based method |
| 1. What is "need"? | Condense nine correlated indicators into a small number of interpretable dimensions; derive a score | Standardisation, log-transform, **PCA** | There is no "need" label. PCA gives a data-driven development axis whose meaning we check via loadings |
| 2. Which types of countries exist? | Group countries by similarity; choose the number of groups; describe them | **K-Means**, inertia / silhouette, cluster profiles | Differentiated programmes require a typology; k is a business judgement supported by metrics |
| 3. Who gets what? | Identify the high-need group, find sub-types inside it, rank within | Second-level clustering, ranking by PCA score, mapping to programmes | Turns analysis into an action; budget forces a cut-off |
| 4. How robust is this? | Stability of clusters across seeds and algorithms; find borderline countries | Adjusted Rand index, **hierarchical clustering**, silhouette per observation | A shortlist that flips with the random seed cannot be defended to a board |


## 8. Reference solution

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

### 8.1 Data audit (question 0)

Before choosing a method we check what we are dealing with: scales, shape of the distributions, extreme values.

In [ ]:
df.describe().T[["min", "50%", "mean", "max"]].round(2).assign(skew=df.skew().round(2))

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 8))
for ax, col in zip(axes.ravel(), df.columns):
    sns.histplot(df[col], bins=30, ax=ax)
    ax.set_title(col); ax.set_xlabel("")
fig.suptitle("Distributions of the nine indicators", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicated rows:", int(df.duplicated().sum()))
print("\nTrade volumes above 100% of GDP (hubs / re-export economies):")
print(df[df.exports > 100][["exports", "imports", "gdpp"]].sort_values("exports", ascending=False))
print("\nNegative inflation:", list(df.index[df.inflation < 0]))
print("\nImplausibly low trade volumes (< 1% of GDP):")
print(df[(df.exports < 1) | (df.imports < 1)][["exports", "imports", "gdpp"]])

**Findings.** Income, GDP per capita, child mortality, exports, imports and especially inflation are strongly right-skewed. Life expectancy is left-skewed.
A few countries have extreme values (trade above 100% of GDP, inflation > 100%) and a few have negative inflation.
**One record looks like a data error:** Myanmar reports exports and imports of about 0.1% of GDP, which is not credible for an economy of that size. The variables
live on very different scales (income in tens of thousands vs. fertility in single digits). Raw K-Means or PCA would be driven by
`income` and `gdpp` alone. We have no missing values, which is unusual and worth a remark.

**Decision.** Log-transform the strongly skewed variables `child_mort`, `income`, `gdpp` and `inflation` (the latter shifted since it contains negative values), then standardise all variables.
We deliberately do **not** log-transform `exports` and `imports`: the log would turn the implausible Myanmar record into an extreme outlier on the low side and push it into a cluster of its own.
We keep all countries in the data (the real extremes, such as trade hubs, are informative), but flag Myanmar as a record to verify with the data provider (see robustness).

### 8.2 Preprocessing

In [ ]:
skewed = ["child_mort", "income", "gdpp", "inflation"]

X = df.copy()
for col in skewed:
    shift = -X[col].min() + 1 if X[col].min() <= 0 else 0   # only inflation needs a shift
    X[col] = np.log(X[col] + shift)

scaler = StandardScaler()
Z = pd.DataFrame(scaler.fit_transform(X), index=df.index, columns=df.columns)
print("Skewness before:\n", df.skew().round(2).to_dict())
print("Skewness after: \n", X.skew().round(2).to_dict())

### 8.3 What is "need"? PCA as a development axis (question 1)

The nine indicators are strongly correlated. PCA reveals how many independent dimensions remain and what they mean.

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(Z.corr(), annot=True, fmt=".1f", cmap="RdBu_r", center=0, cbar=False)
plt.title("Correlation of the preprocessed indicators"); plt.show()

In [ ]:
pca = PCA(random_state=RANDOM_STATE).fit(Z)
evr = pca.explained_variance_ratio_
print("Explained variance ratio:", evr.round(2))
print("Cumulative (first 3):    ", evr.cumsum()[:3].round(2))

loadings = pd.DataFrame(pca.components_[:3].T, index=Z.columns, columns=["PC1", "PC2", "PC3"]).round(2)
loadings

**Interpretation.** The first component alone carries more than half of the variance. It has high loadings on child mortality and
fertility and opposite loadings on life expectancy, income and GDP per capita: a classic **development / need axis**. We orient it so that
a high score means high need. PC2 mainly captures the trade dimension (exports/imports) and PC3 inflation and health spending.
Because the first axis is clearly interpretable, it is a reasonable, transparent **need score**, but we still treat it as a proxy,
not a ground truth.

In [ ]:
sign = np.sign(pca.components_[0][Z.columns.get_loc("child_mort")])
need = pd.Series(pca.transform(Z)[:, 0] * sign, index=Z.index, name="need_score")
df["need_score"] = need

print("Top 15 countries by need score:")
need.sort_values(ascending=False).head(15).round(2)

### 8.4 What types of countries exist? Clustering (question 2)

We compare candidate numbers of clusters via inertia and silhouette, then combine this with business judgement.

In [ ]:
ks = range(2, 11)
inertia, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE).fit(Z)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(Z, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(ks, inertia, "o-"); ax[0].set(title="Elbow plot", xlabel="k", ylabel="Inertia (WCSS)")
ax[1].plot(ks, sil, "o-"); ax[1].set(title="Silhouette", xlabel="k", ylabel="Mean silhouette")
plt.tight_layout(); plt.show()

**Choosing k.** The elbow is not sharp. The silhouette is highest for k = 2 (roughly "poor" vs. "not poor"), which is a valid but not very useful answer:
the foundation has three programme types and wants to see types of countries. From k = 4 onwards the silhouette stays flat while
the inertia continues to fall moderately. We choose **k = 4** and verify that the clusters are interpretable. If they are, the choice is
defensible on business grounds, not only statistical ones.

In [ ]:
K = 4
km = KMeans(n_clusters=K, n_init=50, random_state=RANDOM_STATE).fit(Z)
df["cluster"] = km.labels_
df["need_score"] = need  # from section 8.3; re-assigned so the cell also works if that cell was skipped

profile = df.groupby("cluster")[list(Z.columns) + ["need_score"]].median().round(1)
profile.insert(0, "n_countries", df.groupby("cluster").size())
profile

In [ ]:
# Name clusters from their profile (rules, not hand-labelled numbers, so the names survive a re-run)
names = {}
names[profile["child_mort"].idxmax()] = "1 Priority (high need)"
names[profile["exports"].idxmax()] = "4 Trade hubs"
rest = [c for c in profile.index if c not in names]
names[profile.loc[rest, "gdpp"].idxmax()] = "3 Advanced / upper-middle"
names[[c for c in rest if c not in names][0]] = "2 Emerging"
df["segment"] = df["cluster"].map(names)

pcs = pd.DataFrame(pca.transform(Z)[:, :2] * [sign, 1], index=Z.index, columns=["PC1", "PC2"])
plt.figure(figsize=(9, 6))
sns.scatterplot(data=pcs.join(df["segment"]).sort_values("segment"), x="PC1", y="PC2", hue="segment", s=60, palette="Set1")
for c in ["Niger", "Singapore", "Luxembourg", "Nigeria", "India", "Norway", "Haiti"]:
    plt.annotate(c, pcs.loc[c], fontsize=8, xytext=(4, 4), textcoords="offset points")
plt.xlabel("PC1 (need / development axis, high = high need)"); plt.ylabel("PC2 (trade openness)")
plt.title("Country types in PCA space"); plt.show()

In [ ]:
order = sorted(df["segment"].unique())
fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, col in zip(axes.ravel(), Z.columns):
    sns.boxplot(data=df, x="segment", y=col, order=order, ax=ax, hue="segment", legend=False, palette="Set1")
    ax.set_xlabel(""); ax.set_title(col); ax.tick_params(axis="x", rotation=25)
    if col in ["child_mort", "income", "gdpp", "exports", "imports", "inflation"]:
        ax.set_yscale("log")
plt.tight_layout(); plt.show()

for s in order:
    print(f"{s}: {', '.join(df.index[df.segment == s][:12])} ...")

**Interpretation.**

* **Priority (high need):** very high child mortality and fertility, low life expectancy, very low income. About 45 countries, mostly in Sub-Saharan Africa plus e.g. Afghanistan and Haiti.
* **Emerging:** middle incomes, moderate child mortality (about 20 per 1,000), but higher inflation and fertility than the advanced group (70+ countries, e.g. Brazil, Bangladesh, Algeria).
* **Advanced / upper-middle:** low mortality, long life expectancy, higher health spending.
* **Trade hubs:** five small, rich economies with exports/imports far above 100% of GDP (e.g. Singapore, Luxembourg, Ireland, Malta). They form their own cluster; with fewer clusters they would have been absorbed and have distorted the "rich" group's profile.

For the foundation, the clustering says: **there is a clear high-need block, and it is not the only distinct type**. Funding decisions can be concentrated on the first block.

### 8.5 Who gets what? Sub-types within the priority group (question 3)

The priority segment is still too large and heterogeneous for a single programme. We cluster *within* the segment,
re-using the same standardised indicators, to find sub-types that map to programmes A, B and C.

In [ ]:
pri = df[df.segment == "1 Priority (high need)"].copy()
Zp = Z.loc[pri.index]

for k in (2, 3, 4):
    lab = KMeans(n_clusters=k, n_init=50, random_state=RANDOM_STATE).fit_predict(Zp)
    print(f"k={k}: silhouette = {silhouette_score(Zp, lab):.3f}")

In [ ]:
sub = KMeans(n_clusters=3, n_init=50, random_state=RANDOM_STATE).fit(Zp)
pri["sub"] = sub.labels_
sub_profile = pri.groupby("sub")[list(Z.columns)].median().round(1)
sub_profile.insert(0, "n", pri.groupby("sub").size())
sub_profile

In [ ]:
for s in sorted(pri["sub"].unique()):
    print(f"Sub-type {s}: {', '.join(pri.index[pri['sub'] == s])}\n")

**Interpretation.** The sub-types differ in a way that matches the programme menu:

* one type has the **highest child mortality, highest fertility and lowest GDP per capita**: a **child survival** problem, programme **A**;
* one type has **volatile prices (inflation far above the other types) and somewhat higher income**: an **economic resilience** problem, programme **C**;
* one small type (mostly small island and aid-dependent states) has **very high imports and the highest health spending**, but still high mortality: spending is not translating into outcomes, a **health system** problem, programme **B**.

The silhouette is marginally higher for two sub-types, but the foundation has three programmes and the three-way split yields profiles with a clear meaning, so we use k = 3.
We map the sub-types to programmes by their profile (verified below), not by cluster number.

In [ ]:
# Assign programme by inspecting the sub-profiles: highest child_mort -> A; highest inflation -> C; remainder (high imports / health spending) -> B
prog = {}
prog[sub_profile["child_mort"].idxmax()] = "A · Child survival & primary care"
prog[sub_profile["inflation"].idxmax()] = "C · Economic resilience"
prog[[s for s in sub_profile.index if s not in prog][0]] = "B · Health system strengthening"
pri["programme"] = pri["sub"].map(prog)
pri["need_score"] = need.loc[pri.index]

pd.concat({p: sub_profile.loc[s] for s, p in prog.items()}, axis=1).T

**Shortlist rule (transparent and explainable to the board).** The budget allows 15 countries. We
1. rank all countries of the priority segment by **need score**, and
2. take the highest-scoring ones while ensuring that each programme type is represented by at least 2 countries (a portfolio, not a single bet).

Note the price of the portfolio rule: the last place goes to a country with a clearly lower need score than the first countries left out of type A. This is a policy choice that the board should make explicitly.

In [ ]:
N, MIN_PER_PROGRAMME = 15, 2

ranked = pri.sort_values("need_score", ascending=False)
guaranteed = ranked.groupby("programme").head(MIN_PER_PROGRAMME)
rest = ranked.drop(guaranteed.index).head(N - len(guaranteed))
shortlist = pd.concat([guaranteed, rest]).sort_values("need_score", ascending=False)

shortlist[["programme", "need_score", "child_mort", "life_expec", "total_fer", "gdpp"]].round(2)

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(pcs.PC1, pcs.PC2, c="lightgrey", s=25, label="other countries")
sns.scatterplot(data=pcs.loc[shortlist.index].join(shortlist["programme"]), x="PC1", y="PC2", hue="programme", s=90, palette="Dark2", edgecolor="black")
for c in shortlist.index:
    plt.annotate(c, pcs.loc[c], fontsize=7, xytext=(4, 3), textcoords="offset points")
plt.xlabel("PC1 (need, high = high need)"); plt.ylabel("PC2 (trade openness)")
plt.title("First-wave shortlist by programme type"); plt.legend(loc="lower left"); plt.show()

### 8.6 How robust is this? (question 4)

We ask three questions: do the clusters change with the random seed, do they change with a different algorithm, and which countries sit on the border?

In [ ]:
ref = km.labels_
seed_ari = [adjusted_rand_score(ref, KMeans(n_clusters=K, n_init=1, random_state=s).fit_predict(Z)) for s in range(30)]
ward = AgglomerativeClustering(n_clusters=K, linkage="ward").fit_predict(Z)

print(f"K-Means, single-start runs vs. reference: ARI median = {np.median(seed_ari):.2f}, min = {np.min(seed_ari):.2f}")
print(f"Ward hierarchical vs. K-Means:             ARI = {adjusted_rand_score(ref, ward):.2f}")

pri_ward = set(df.index[ward == pd.Series(ward, index=df.index)[df.index[df.segment == '1 Priority (high need)']].mode()[0]])
pri_km = set(pri.index)
print(f"\nPriority group: K-Means {len(pri_km)} countries, Ward group {len(pri_ward)} countries, overlap {len(pri_km & pri_ward)}")
print("Shortlisted countries outside the Ward priority group:", sorted(set(shortlist.index) - pri_ward) or "none")

In [ ]:
df["silhouette"] = silhouette_samples(Z, df["cluster"])
border = df[(df.segment == "1 Priority (high need)")].sort_values("silhouette").head(8)[["silhouette", "child_mort", "total_fer", "gdpp"]]
print("Least clearly assigned countries of the priority segment:")
print(border.round(2))
print("\nShortlisted countries with a silhouette below 0.1:", list(shortlist.index[df.loc[shortlist.index, "silhouette"] < 0.1]) or "none")

**Reading the results.**

* **Seeds.** Re-running K-Means from different random starts gives similar but not identical partitions: the clusters are real, but their boundaries are soft. We therefore use many restarts (`n_init=50`) for the final model.
* **Algorithm.** Ward clustering agrees only moderately overall (ARI around 0.5), but the disagreement is about *where the high-need block ends*: Ward's version is wider and absorbs the borderline mid-range countries, while it contains all K-Means priority countries. Nothing on our shortlist depends on this boundary.
* **Shortlist.** The countries at the top of the need ranking are the same in each view. The uncertainty concerns the **lower-ranked** shortlist places and the programme assignment (the sub-types have weaker separation).
* **Borderline countries** (low silhouette, e.g. Cambodia, Lao, Nepal) are not on the shortlist but would be the first to be argued about. They should be flagged as "decision needs additional information", e.g. a country report.

**Implication for the board:** the top of the list is robust, the programme types for borderline countries are a judgement call, not a result.

### 8.7 Recommendation (template for the final slide)

| | |
|---|---|
| **Need** | A development axis (PC1) explains more than half of the variation across nine indicators and is dominated by child mortality, fertility, life expectancy and income. |
| **Types** | Four country types exist: high-need, emerging, advanced and trade hubs. Only the first is relevant for the foundation's mission. |
| **Shortlist** | 15 countries from the high-need group, see table above, mixing programmes A, B and C. |
| **Confidence** | The high-need group is stable across seeds and algorithms; the programme assignment and countries ranked at the bottom of the shortlist are less certain. |

**Limitations to state.**
1. Country averages hide inequality inside countries (e.g. regions in large countries such as Nigeria or India).
2. The data measures *need*, not *impact per euro*: governance, conflict, absorptive capacity and other donors are missing.
3. A snapshot of ca. 2010 to 2015: the situation may have changed; trends are not captured.
4. Clusters are a modelling choice (k, scaling, algorithm), not natural facts. PCA is linear and the need score assumes equal importance of the underlying drivers.
5. Programme mapping rests on a handful of indicators; local expertise is needed before the grant.

**Next steps.** Add trend data, governance and conflict indicators, and validate programme assignment with regional experts.